# Statistical Machine Learning Course Project
## Text Classification: Sparse Features vs Contextual Embeddings

Before running: open `Runtime > Change runtime type`, choose a GPU such as T4, and save.

## Step 0: Check GPU

In [ ]:
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 1: Set Up Project Directory

**Option A (Recommended - Direct Git Clone):**
```python
!git clone https://github.com/tharunsrisanth07-dotcom/ML-project.git
%cd ML-project
```

**Option B (Upload Zip):**
Run the cell below and upload a zip of the project. If already cloned or present, it will automatically detect the files.


In [ ]:
import os
import zipfile

# Check if project files already exist in current directory or cloned location
if os.path.exists("config.py") and os.path.exists("main.py"):
    print("Found project in current directory:", os.getcwd())
elif os.path.exists("/content/ML-project/config.py"):
    os.chdir("/content/ML-project")
    print("Moved to cloned directory:", os.getcwd())
else:
    from google.colab import files
    print("Upload your project zip file (or run: !git clone https://github.com/tharunsrisanth07-dotcom/ML-project.git):")
    uploaded = files.upload()

    extract_root = "/content/project_files"
    os.makedirs(extract_root, exist_ok=True)

    for fname in uploaded.keys():
        if fname.lower().endswith(".zip"):
            with zipfile.ZipFile(fname, "r") as zf:
                zf.extractall(extract_root)
            print(f"Extracted: {fname}")

    project_dir = None
    for root, dirs, fnames in os.walk(extract_root):
        if "config.py" in fnames and "main.py" in fnames:
            project_dir = root
            break

    if project_dir is None:
        raise FileNotFoundError("Could not find config.py and main.py in the uploaded zip.")

    os.chdir(project_dir)

print("Working directory:", os.getcwd())
print("Project files:")
for name in sorted(os.listdir(".")):
    print("  " + name)


## Step 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

Optional: install TeX only if you want Colab to compile `report.pdf`. The LaTeX source is generated without this.

In [ ]:
# Optional PDF support. This can take a few minutes.
# !apt-get update -y >/dev/null
# !apt-get install -y -qq texlive-latex-recommended texlive-fonts-recommended texlive-latex-extra >/dev/null

## Stage 1-2: Load Dataset and EDA

In [ ]:
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import config
from data_loader import load_and_sample_dataset, run_eda, split_dataset

df = load_and_sample_dataset()
run_eda(df)

## Stage 3-4: Train/Test Split and Text Preprocessing

In [ ]:
from preprocessing import TextPreprocessor

X_train_raw, X_test_raw, y_train, y_test = split_dataset(df)

preprocessor = TextPreprocessor()
X_train_clean = preprocessor.preprocess(X_train_raw)
X_test_clean = preprocessor.preprocess(X_test_raw)

print(f'Preprocessed {len(X_train_clean)} train + {len(X_test_clean)} test texts')
print('Before:', X_train_raw[0][:120])
print('After: ', X_train_clean[0][:120])

## Stage 5: Sparse Feature Extraction

In [ ]:
from feature_extraction.sparse_features import SparseFeatureExtractor

sparse_extractor = SparseFeatureExtractor()
sparse_train = sparse_extractor.fit_transform(X_train_clean)
sparse_test = sparse_extractor.transform(X_test_clean)

## Stage 6: Frozen Embeddings

DistilBERT uses the GPU if available. GloVe downloads a large file the first time, then caches vectors and document embeddings.

In [ ]:
from feature_extraction.contextual_embeddings import DistilBERTExtractor, GloVeExtractor

print('=' * 50)
print('DistilBERT embeddings')
print('=' * 50)
distilbert = DistilBERTExtractor()
distilbert_train = distilbert.extract_embeddings(X_train_raw, cache_prefix='distilbert_train')
distilbert_test = distilbert.extract_embeddings(X_test_raw, cache_prefix='distilbert_test')

print('\n' + '=' * 50)
print('GloVe embeddings')
print('=' * 50)
glove = GloVeExtractor()
glove_train = glove.extract_embeddings(X_train_clean, cache_prefix='glove_train')
glove_test = glove.extract_embeddings(X_test_clean, cache_prefix='glove_test')

## Stage 7: Train Classical ML Models

In [ ]:
from tqdm import tqdm
from models.classifiers import train_and_evaluate_all

all_features = {
    'BoW': {'train': sparse_train['bow'], 'test': sparse_test['bow']},
    'TF-IDF': {'train': sparse_train['tfidf'], 'test': sparse_test['tfidf']},
    'GloVe': {'train': glove_train['glove'], 'test': glove_test['glove']},
    'DistilBERT-CLS': {
        'train': distilbert_train['distilbert_cls'],
        'test': distilbert_test['distilbert_cls'],
    },
    'DistilBERT-Mean': {
        'train': distilbert_train['distilbert_mean'],
        'test': distilbert_test['distilbert_mean'],
    },
}

for name, data in all_features.items():
    print(f'{name}: train={data["train"].shape}, test={data["test"].shape}')

all_results = {}
for feat_name, feat_data in tqdm(all_features.items(), desc='Feature types'):
    print(f'\n=== Feature: {feat_name} ===')
    all_results[feat_name] = train_and_evaluate_all(
        feat_data['train'], feat_data['test'], y_train, y_test, feature_name=feat_name
    )

## Stage 8: Dimensionality Reduction

In [ ]:
from dimensionality_reduction import analyze_dimensionality_reduction

dr_features = {
    'TF-IDF': {'train': sparse_train['tfidf'], 'test': sparse_test['tfidf'], 'is_sparse': True},
    'GloVe': {'train': glove_train['glove'], 'test': glove_test['glove'], 'is_sparse': False},
    'DistilBERT-CLS': {
        'train': distilbert_train['distilbert_cls'],
        'test': distilbert_test['distilbert_cls'],
        'is_sparse': False,
    },
    'DistilBERT-Mean': {
        'train': distilbert_train['distilbert_mean'],
        'test': distilbert_test['distilbert_mean'],
        'is_sparse': False,
    },
}

dr_results = analyze_dimensionality_reduction(dr_features, y_train, y_test)

## Stage 9: Evaluation

In [ ]:
from evaluation import evaluate_all_results

metrics_df, detailed_metrics = evaluate_all_results(all_results, y_test)

## Stage 10: Visualizations

In [ ]:
import visualization as viz

tsne_features = {
    'BoW': sparse_test['bow'],
    'TF-IDF': sparse_test['tfidf'],
    'GloVe': glove_test['glove'],
    'DistilBERT-CLS': distilbert_test['distilbert_cls'],
    'DistilBERT-Mean': distilbert_test['distilbert_mean'],
}

print('Generating t-SNE plots. This can take a few minutes.')
tsne_results = viz.generate_all_tsne_plots(tsne_features, y_test)

In [ ]:
print('Generating dimensionality reduction plots.')
viz.plot_explained_variance(dr_results)
viz.plot_scree(dr_results)
viz.plot_performance_vs_components(dr_results)

print('Generating model comparison plots.')
viz.plot_model_comparison_bars(metrics_df)
viz.plot_heatmap_results(metrics_df)
viz.plot_training_time(metrics_df)
viz.plot_per_class_f1(detailed_metrics)

In [ ]:
print('Generating confusion matrices.')
viz.plot_confusion_matrices(detailed_metrics, y_test)

print('Generating radar charts.')
for feat_name in ['TF-IDF', 'DistilBERT-CLS']:
    if feat_name in detailed_metrics:
        viz.plot_radar_chart(detailed_metrics, feat_name)

print('Generating KMeans cluster plots.')
for feat_name in ['TF-IDF', 'DistilBERT-CLS']:
    if feat_name in all_results and feat_name in tsne_results:
        kmeans_result = all_results[feat_name].get('KMeans', {})
        if 'cluster_labels' in kmeans_result:
            viz.plot_kmeans_clusters(
                tsne_results[feat_name], kmeans_result['cluster_labels'], y_test, feat_name
            )

## Display Saved Plots Again

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob('plots/*.png')):
    print(path)
    display(Image(filename=path))

## Generate Report

In [ ]:
!python generate_report.py --author "Your Name"

with open('results/comparison_summary.txt', 'r', encoding='utf-8') as f:
    print(f.read())

## View Results Tables

In [ ]:
import pandas as pd

results_table = pd.read_csv('results/comparison_results.csv')
display(results_table.sort_values('Accuracy', ascending=False))

kmeans_table = pd.read_csv('results/kmeans_clustering_metrics.csv')
display(kmeans_table)

## Download Outputs

In [ ]:
import os
import zipfile
from google.colab import files

zip_name = 'ML_Project_Complete.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['plots', 'results']:
        for root, dirs, fnames in os.walk(folder):
            for fname in fnames:
                path = os.path.join(root, fname)
                zf.write(path)

    for root_file in os.listdir('.'):
        if root_file.endswith(('.py', '.txt', '.md', '.ipynb')):
            zf.write(root_file)

    for folder in ['feature_extraction', 'models']:
        for root, dirs, fnames in os.walk(folder):
            for fname in fnames:
                if fname.endswith('.py'):
                    path = os.path.join(root, fname)
                    zf.write(path)

print(f'Created {zip_name}')
files.download(zip_name)